# 🚦 Template Notebook: Google Colab & Kaggle
## Hệ Thống Nhận Diện Tai Nạn Giao Thông Từ Video

Notebook này là template chuẩn hóa dùng chung cho cả **Google Colab** và **Kaggle Notebooks**, phục vụ suốt lộ trình dự án (Tuần 1 đến Tuần 8).

### Tính năng tích hợp sẵn:
1. **Tự động nhận diện môi trường** (Google Colab vs Kaggle).
2. **Mount Google Drive** và tự động khởi tạo cấu trúc thư mục lưu trữ chuẩn.
3. **Cài đặt dependencies**: `ultralytics`, `pytorchvideo`, `opencv-python`, `scipy`, `scikit-learn`, `focal-loss-torch`.
4. **Clone / Sync Git repo** từ GitHub (hỗ trợ cả Repo Public lẫn Private qua Token).
5. **Quản lý Checkpoints**: Tự động tìm checkpoint gần nhất, hỗ trợ resume training và backup kết quả về Google Drive tránh mất dữ liệu khi hết session.

--- 
## 1. Nhận diện môi trường & Mount Google Drive

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# 1. Phát hiện môi trường thực thi
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    print("🔵 Đang chạy trên Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Đường dẫn lưu trữ bền vững trên Google Drive
    PROJECT_BASE = Path("/content/drive/MyDrive/accident_detection")
    WORKSPACE_DIR = Path("/content")
elif IS_KAGGLE:
    print("🟠 Đang chạy trên Kaggle")
    # Trên Kaggle, output lưu tại /kaggle/working
    PROJECT_BASE = Path("/kaggle/working/accident_detection")
    WORKSPACE_DIR = Path("/kaggle/working")
else:
    print("💻 Đang chạy trên Local/Khác")
    PROJECT_BASE = Path("./storage")
    WORKSPACE_DIR = Path(".")

# 2. Khởi tạo cấu trúc thư mục lưu trữ chuẩn theo Kế hoạch triển khai (Mục 5)
CHECKPOINT_DIR = PROJECT_BASE / "checkpoints"
DATASET_DIR = PROJECT_BASE / "datasets"
RESULTS_DIR = PROJECT_BASE / "results"
LOGS_DIR = PROJECT_BASE / "logs"
EXPORTS_DIR = PROJECT_BASE / "exports"

for path in [CHECKPOINT_DIR, DATASET_DIR, RESULTS_DIR, LOGS_DIR, EXPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"\n✅ Đã thiết lập thư mục làm việc:")
print(f"   - Project Storage: {PROJECT_BASE}")
print(f"   - Checkpoints:     {CHECKPOINT_DIR}")
print(f"   - Datasets:        {DATASET_DIR}")

--- 
## 2. Kiểm tra phần cứng & GPU

In [ ]:
import torch

print("=== KIỂM TRA PHẦN CỨNG ===")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    device_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"Device: {device_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Chưa kích hoạt GPU! Hãy đổi Runtime sang GPU (T4 hoặc P100).")

# Kiểm tra nvidia-smi
!nvidia-smi

--- 
## 3. Clone / Update Repository từ GitHub (Tự Động Bắt Lỗi & Hỗ Trợ Private Repo)

In [ ]:
import subprocess

REPO_NAME = "Traffic-Accident-Detection"
REPO_PATH = WORKSPACE_DIR / REPO_NAME

# ⚠️ NẾU REPO CỦA BẠN LÀ PRIVATE:
# Hãy điền GitHub Personal Access Token (PAT) vào biến GITHUB_TOKEN dưới đây.
# Nếu repo là PUBLIC, hãy để GITHUB_TOKEN = ""
GITHUB_TOKEN = ""  # Ví dụ: "ghp_xxxxxx..."

if GITHUB_TOKEN:
    REPO_URL = f"https://github.com/ThanhND2005/Traffic-Accident-Detection"
else:
    REPO_URL = f"https://github.com/ThanhND2005/Traffic-Accident-Detection"

print(f"🔍 Đang kiểm tra repository tại: {REPO_PATH}")

if not REPO_PATH.exists():
    print(f"🔄 Đang clone repo từ GitHub...")
    cmd = f"git clone {REPO_URL} {REPO_PATH}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode != 0:
        print("\n❌ LỖI KHI CLONE REPO:")
        print(result.stderr)
        print("-" * 60)
        print("👉 NGUYÊN NHÂN VÀ CÁCH KHẮC PHỤC:")
        print("1. KIỂM TRA INTERNET (Trên Kaggle):")
        print("   - Nhìn sang panel bên phải: 'Settings' -> 'Internet' phải đang bật ON.")
        print("   - Nếu chưa có nút Internet ON, bạn cần xác thực SĐT tại kaggle.com/settings.")
        print("2. NẾU REPO LÀ PRIVATE:")
        print("   - Cách nhanh nhất: Vào github.com/ThanhND2005/Traffic-Accident-Detection -> Settings -> Đổi sang PUBLIC.")
        print("   - Hoặc: Tạo GitHub Token (classic) có quyền repo, rồi gán vào biến GITHUB_TOKEN ở trên.")
        print("-" * 60)
        raise RuntimeError("Clone repo thất bại. Hãy kiểm tra Internet hoặc quyền truy cập Private repo!")
    else:
        print("✅ Clone repo thành công!")
else:
    print(f"🔄 Repo đã tồn tại. Đang pull cập nhật mới nhất...")
    !cd {REPO_PATH} && git pull origin main

# Chuyển thư mục làm việc và thêm vào sys.path
os.chdir(REPO_PATH)
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))

print(f"\n✅ Đang đứng tại thư mục: {os.getcwd()}")

--- 
## 4. Cài đặt các thư viện phụ thuộc (Dependencies)

In [ ]:
# Cài đặt các gói theo yêu cầu của kế hoạch triển khai
!pip install -q ultralytics opencv-python scipy scikit-learn focal-loss-torch pyyaml

# Cài đặt PyTorchVideo (cần thiết cho nhánh Visual X3D ở Tuần 7)
try:
    import pytorchvideo
    print("PyTorchVideo đã có sẵn.")
except ImportError:
    print("Đang cài đặt PyTorchVideo từ Facebook Research...")
    !pip install -q fvcore iopath
    !pip install -q "git+https://github.com/facebookresearch/pytorchvideo.git"

# Kiểm tra các import cốt lõi
import ultralytics
from ultralytics import YOLO
import cv2
import sklearn
import scipy

print("\n✅ Cài đặt dependencies thành công!")
print(f"   - Ultralytics version: {ultralytics.__version__}")
print(f"   - OpenCV version:      {cv2.__version__}")
print(f"   - Scikit-learn:        {sklearn.__version__}")
print(f"   - SciPy:               {scipy.__version__}")

--- 
## 5. Tiện ích quản lý Checkpoints (Load, Resume & Backup)

In [ ]:
def list_drive_checkpoints():
    """Liệt kê tất cả file trọng số đã lưu trữ."""
    print(f"📂 Danh sách checkpoints tại: {CHECKPOINT_DIR}")
    files = list(CHECKPOINT_DIR.rglob("*.pt")) + list(CHECKPOINT_DIR.rglob("*.pth"))
    if not files:
        print("   (Chưa có checkpoint nào được lưu trữ)")
    else:
        for f in files:
            size_mb = f.stat().st_size / (1024 * 1024)
            rel_path = f.relative_to(CHECKPOINT_DIR)
            print(f"   - {rel_path} ({size_mb:.2f} MB)")
    return files

def get_checkpoint_path(module_name="yolo11s_vn_traffic", prefer="best"):
    """
    Tìm đường dẫn checkpoint theo module:
    module_name: 'yolo11s_vn_traffic', 'lstm_trajectory', 'x3d_accident', 'fusion'
    prefer: 'best' hoặc 'last'
    """
    candidate_dir = CHECKPOINT_DIR / module_name
    if not candidate_dir.exists():
        candidate_dir = CHECKPOINT_DIR
        
    target = candidate_dir / f"{prefer}.pt"
    if target.exists():
        print(f"✅ Tìm thấy checkpoint: {target}")
        return str(target)
    
    matches = list(candidate_dir.glob(f"*{prefer}*.pt"))
    if matches:
        print(f"✅ Tìm thấy checkpoint khớp tên: {matches[0]}")
        return str(matches[0])
    
    print(f"ℹ️ Chưa có checkpoint '{prefer}' cho {module_name}. Sẽ sử dụng pretrained hoặc train từ đầu.")
    return None

def backup_checkpoint(source_path, module_name="yolo11s_vn_traffic", checkpoint_name="best.pt"):
    """Sao chép checkpoint về thư mục lưu trữ checkpoints."""
    src = Path(source_path)
    if not src.exists():
        print(f"❌ Không tìm thấy file nguồn: {src}")
        return
    
    target_dir = CHECKPOINT_DIR / module_name
    target_dir.mkdir(parents=True, exist_ok=True)
    dest = target_dir / checkpoint_name
    
    shutil.copy(src, dest)
    print(f"💾 Đã lưu checkpoint thành công:")
    print(f"   Source: {src}")
    print(f"   Destination: {dest} ({dest.stat().st_size / (1024*1024):.2f} MB)")

# Kiểm tra danh sách checkpoint hiện tại
list_drive_checkpoints()

--- 
## 6. Kiểm tra Import Modules dự án từ thư mục `src/`

In [ ]:
# Kiểm tra import các module chuẩn từ src/ của repo
try:
    from src.detection.yolo_detector import YOLODetector
    from src.tracking.trajectory_manager import TrajectoryManager
    from src.features.motion_features import MotionFeatureExtractor
    from src.classifiers.rule_based import RuleBasedAccidentDetector
    print("✅ Import thành công các module cốt lõi từ thư mục src/!")
except ImportError as e:
    print(f"⚠️ Lưu ý khi import: {e}")
    print("Kiểm tra xem bạn đã clone đúng repo và thư mục src/ có tồn tại hay không.")